In [1]:
import pandas as pd
import numpy as np  

In [2]:
df = pd.read_excel('../raw/data/Online Retail.xlsx')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [4]:
df.shape

(541909, 8)

In [5]:
df.isna().sum() 

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(5268)

In [7]:
df["InvoiceNo"].str.startswith("C").sum()

9288

In [8]:
(df["Quantity"] < 0).sum()

np.int64(10624)

In [9]:
(df["UnitPrice"] <= 0).sum()

np.int64(2517)

In [10]:
df["CustomerID"].nunique()

4372

In [11]:
df["InvoiceDate"].min(), df["InvoiceDate"].max()

(Timestamp('2010-12-01 08:26:00'), Timestamp('2011-12-09 12:50:00'))

In [12]:
import pandas as pd
import numpy as np

def detectar_nulos_vectorizado(df: pd.DataFrame, valores_extra: list = None) -> pd.DataFrame:
    """
    Detección de nulos vectorizada (barata para datasets grandes).
    Cubre NaN/None reales + strings vacíos/espacios + strings tipo NA/null.
    """
    valores_nulos_str = {
        'na', 'n/a', 'null', 'none', 'nan', 'nil', '-', '--', '?',
        's/d', 'sin dato', 'sin datos', 'undefined'
    }
    if valores_extra:
        valores_nulos_str.update(v.lower() for v in valores_extra)

    mask_real = df.isna()

    cols_texto = df.select_dtypes(include=['object', 'string']).columns
    mask_str = pd.DataFrame(False, index=df.index, columns=df.columns)

    if len(cols_texto) > 0:
        # .map en vez de .str: tolera columnas 'object' con tipos mixtos
        normalizado = df[cols_texto].apply(
            lambda s: s.map(lambda v: v.strip().lower() if isinstance(v, str) else v)
        )
        mask_str[cols_texto] = normalizado.isin(valores_nulos_str) | (normalizado == '')

    mask_nulos = mask_real | mask_str

    filas = []
    for col in df.columns:
        n_nulos = mask_nulos[col].sum()
        if n_nulos == 0:
            continue

        formas = set()
        if mask_real[col].any():
            formas.add('NaN')
        if col in cols_texto and mask_str[col].any():
            crudos = df.loc[mask_str[col] & ~mask_real[col], col].astype(str).str.strip().unique()
            formas.update(crudos.tolist())

        formas = sorted(formas, key=str)
        valor_repr = formas[0] if len(formas) == 1 else formas

        filas.append({
            'columna': col,
            'nulos': int(n_nulos),
            'porcentaje_nulos': round(n_nulos / len(df) * 100, 2),
            'valor': valor_repr
        })

    resumen = pd.DataFrame(filas).sort_values('nulos', ascending=False).reset_index(drop=True)
    return resumen

In [13]:
detectar_nulos_vectorizado(df)

,columna,nulos,porcentaje_nulos,valor
0,CustomerID,135080,24.93,NaN
1,Description,1501,0.28,"[?, NaN]"


In [14]:
cancelled = df[df["InvoiceNo"].astype(str).str.startswith("C")]
cancelled.shape

(9288, 8)

In [15]:
cancelled[["InvoiceNo", "Quantity", "UnitPrice", "CustomerID"]].head(10)

,InvoiceNo,Quantity,UnitPrice,CustomerID
141,C536379,-1,27.50,14527.0
154,C536383,-1,4.65,15311.0
235,C536391,-12,1.65,17548.0
236,C536391,-24,0.29,17548.0
237,C536391,-24,0.29,17548.0
238,C536391,-24,0.29,17548.0
239,C536391,-12,3.45,17548.0
240,C536391,-12,1.65,17548.0
241,C536391,-24,1.65,17548.0
939,C536506,-6,4.25,17897.0


In [16]:
(cancelled["Quantity"] < 0).value_counts()

Quantity
True    9288
Name: count, dtype: int64

In [17]:
(df["Quantity"] < 0).value_counts()

Quantity
False    531285
True      10624
Name: count, dtype: int64

In [18]:
df_quantity = df[df["Quantity"] < 0]
df_quantity =  df_quantity[~(df_quantity["InvoiceNo"].astype(str).str.startswith("C"))]
df_quantity.shape

(1336, 8)

In [19]:

df_quantity.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom
4347,536764,84952C,NaN,-38,2010-12-02 14:42:00,0.0,NaN,United Kingdom
7188,536996,22712,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7189,536997,22028,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7190,536998,85067,NaN,-6,2010-12-03 15:30:00,0.0,NaN,United Kingdom


In [20]:
(df_quantity["UnitPrice"] == 0).value_counts()
df_quantity["CustomerID"].isna().sum()
df_quantity["Description"]

2406          NaN
4347          NaN
7188          NaN
7189          NaN
7190          NaN
           ...   
535333      check
535335       lost
535336      check
536908    missing
538919    smashed
Name: Description, Length: 1336, dtype: object

In [21]:
df_quantity["Description"].isna().sum()

np.int64(862)

In [22]:
detectar_nulos_vectorizado(df_quantity) 

,columna,nulos,porcentaje_nulos,valor
0,CustomerID,1336,100.00,NaN
1,Description,903,67.59,"[?, NaN]"


In [23]:
df_quantity[ df_quantity["Description"].notna() ]["Description"].value_counts()

Description
check                    120
damages                   45
damaged                   42
?                         41
sold as set on dotcom     20
                        ... 
lost??                     1
wet                        1
wet boxes                  1
????damages????            1
lost                       1
Name: count, Length: 138, dtype: int64

In [24]:
df.loc[(df['UnitPrice']<=0) &  (~df['InvoiceNo'].astype(str).str.startswith("C"))].shape

(2517, 8)

In [26]:
df["StockCode"].map(type).value_counts()

StockCode
<class 'int'>    487036
<class 'str'>     54873
Name: count, dtype: int64

dtype('O')